# Regresión Logística

In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("data/master_dataset.csv")

In [4]:
leakage_cols = ["omdb_oscar_wins", "omdb_awards", "days_to_ceremony"]


In [5]:
# Features absolutos

abs_features = [
    "imdb_rating", "rt_score", "metacritic", "tmdb_vote_avg",
    "tmdb_popularity", "log_imdb_votes", "critic_composite",
    "budget_m", "revenue_m", "roi",
    "runtime_min", "is_q4_release", "is_english", "n_nominees_year",
    "total_precursor_wins", "total_precursor_noms",
    "BAFTA_best_film_won", "GG_drama_won", "GG_comedy_won",
    "CCA_best_picture_won", "PGA_best_picture_won",
    "genre_drama", "genre_biography", "genre_history",
    "genre_romance", "genre_thriller", "genre_war",
]

In [6]:
# Relativos al año

rel_features = [f"{f}_pct_year" for f in [
    "imdb_rating", "rt_score", "metacritic", "tmdb_vote_avg",
    "tmdb_popularity", "budget_m", "revenue_m",
    "total_precursor_wins", "critic_composite"
]] + [
    "imdb_rating_is_max", "rt_score_is_max", "metacritic_is_max",
    "total_precursor_wins_is_max", "is_precursor_leader",
    "critic_composite_is_max",
]

In [7]:
all_features = [f for f in abs_features + rel_features
                if f in df.columns and f not in leakage_cols]

In [8]:
train_years = list(range(2005, 2019))
val_years   = list(range(2019, 2022))
test_years  = list(range(2022, 2026))

df_train    = df[df["ceremony_year"].isin(train_years)]
df_val      = df[df["ceremony_year"].isin(val_years)]
df_test     = df[df["ceremony_year"].isin(test_years)]
df_trainval = pd.concat([df_train, df_val])

X_train = df_train[all_features].fillna(-999)
y_train = df_train["won_best_picture"]

In [9]:
# ── Escalar features (necesario para LR) ─────────────────────────────
scaler  = StandardScaler()
X_train_scaled = scaler.fit_transform(df_train[all_features].fillna(-999))
X_val_scaled   = scaler.transform(df_val[all_features].fillna(-999))
X_test_scaled  = scaler.transform(df_test[all_features].fillna(-999))

In [10]:
# ── Tuning C con val set ──────────────────────────────────────────────
best_c, best_acc_lr = None, 0
for C in [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]:
    lr = LogisticRegression(C=C, class_weight="balanced",
                            max_iter=1000, random_state=42)
    lr.fit(X_train_scaled, df_train["won_best_picture"])
    
    scores = []
    for year in val_years:
        mask       = df_val["ceremony_year"] == year
        probs      = lr.predict_proba(X_val_scaled[mask])[:, 1]
        probs_norm = probs / probs.sum()
        winner_prob  = probs_norm[df_val.loc[df_val["ceremony_year"] == year,
                                             "won_best_picture"].values == 1][0]
        scores.append((probs_norm < winner_prob).mean())
    
    acc = np.mean(scores)
    if acc > best_acc_lr:
        best_acc_lr = acc
        best_c      = C

print(f"Mejor C: {best_c}  |  Val score: {best_acc_lr:.3f}")

Mejor C: 0.01  |  Val score: 0.764


In [11]:
# ── Reentrenar con train+val ──────────────────────────────────────────
X_trainval_scaled = scaler.fit_transform(df_trainval[all_features].fillna(-999))
X_test_scaled_f   = scaler.transform(df_test[all_features].fillna(-999))

lr_final = LogisticRegression(C=best_c, class_weight="balanced",
                               max_iter=1000, random_state=42)
lr_final.fit(X_trainval_scaled, df_trainval["won_best_picture"])

# ── Test ──────────────────────────────────────────────────────────────
print("\n── Logistic Regression — Test Set (2022-2025) ───────────────")
lr_results = []
for year in test_years:
    mask       = df_test["ceremony_year"] == year
    year_df    = df_test[mask].copy()
    probs      = lr_final.predict_proba(X_test_scaled_f[mask.values])[:, 1]
    probs_norm = probs / probs.sum()
    year_df["prob_lr"] = probs_norm
    
    pred  = year_df.sort_values("prob_lr", ascending=False).iloc[0]["nominated_title"]
    real  = year_df[year_df["won_best_picture"] == 1]["nominated_title"].values[0]
    wprob = year_df[year_df["won_best_picture"] == 1]["prob_lr"].values[0]
    ok    = int(pred == real)
    
    print(f"{year} | Real: {real:<40} Pred: {pred:<40} {wprob:.1%} {'✅' if ok else '❌'}")
    lr_results.append({"year": year, "real": real, "pred": pred,
                       "correct": ok, "winner_prob": wprob, "model": "LogReg"})

lr_df = pd.DataFrame(lr_results)
print(f"\nAccuracy LR: {lr_df['correct'].mean():.1%}")


── Logistic Regression — Test Set (2022-2025) ───────────────
2022 | Real: CODA                                     Pred: The Power of the Dog                     16.9% ❌
2023 | Real: Everything Everywhere All at Once        Pred: Everything Everywhere All at Once        21.4% ✅
2024 | Real: Oppenheimer                              Pred: Oppenheimer                              22.4% ✅
2025 | Real: Anora                                    Pred: Anora                                    19.0% ✅

Accuracy LR: 75.0%


In [12]:
import joblib
joblib.dump(lr_final, "models/logreg_oscar.pkl")
joblib.dump(scaler,   "models/scaler.pkl")

['models/scaler.pkl']